---
## Membaca Dataset

Jalankan ulang cell pembuatan `SparkSession` dan kedua DataFrame (`df_transaksi`, `df_produk`) dari awal modul sebelum mengerjakan latihan berikut.

In [34]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, rank, row_number, when
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Pertemuan5-JoinWindowSQL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


Pada pertemuan ini kita akan bekerja dengan **dua tabel** sekaligus — mensimulasikan skenario dunia nyata di mana data tersebar di beberapa sumber dan perlu digabungkan (persis seperti *join* pada SQL/database relasional):

1. **`df_transaksi`** — data transaksi (seperti pertemuan-pertemuan sebelumnya)
2. **`df_produk`** — tabel referensi/master berisi target penjualan bulanan & nama manager per kategori produk

In [35]:
import numpy as np
import pandas as pd

np.random.seed(7)
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]

# Tabel referensi/master: target & manager per kategori (data ini relatif statis, jarang berubah)
data_produk = {
    "kategori": kategori_list,
    "target_bulanan": [50000000, 40000000, 30000000, 25000000, 20000000],
    "manager": ["Andi", "Budi", "Citra", "Dewi", "Eka"],
}
df_produk = spark.createDataFrame(pd.DataFrame(data_produk))

# Tabel transaksi: data yang terus bertambah setiap hari
n = 300
data_transaksi = {
    "order_id": [f"O{i}" for i in range(n)],
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(["Magelang", "Semarang", "Solo"], size=n),
    "pendapatan": np.random.randint(50000, 500000, size=n),
}
df_transaksi = spark.createDataFrame(pd.DataFrame(data_transaksi))

print("df_produk:")
df_produk.show()
print("df_transaksi (5 baris pertama dari total", df_transaksi.count(), "baris):")
df_transaksi.show(5)

df_produk:


+--------------------+--------------+-------+
|            kategori|target_bulanan|manager|
+--------------------+--------------+-------+
|          Elektronik|      50000000|   Andi|
|             Fashion|      40000000|   Budi|
|   Makanan & Minuman|      30000000|  Citra|
|Kesehatan & Kecan...|      25000000|   Dewi|
|        Rumah Tangga|      20000000|    Eka|
+--------------------+--------------+-------+



df_transaksi (5 baris pertama dari total 300 baris):
+--------+--------------------+--------+----------+
|order_id|            kategori|    kota|pendapatan|
+--------+--------------------+--------+----------+
|      O0|        Rumah Tangga|Magelang|    488643|
|      O1|             Fashion|Magelang|    401943|
|      O2|Kesehatan & Kecan...|    Solo|    452308|
|      O3|Kesehatan & Kecan...|Semarang|    421741|
|      O4|        Rumah Tangga|Semarang|    185244|
+--------+--------------------+--------+----------+
only showing top 5 rows



## Menutup SparkSession

In [36]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.


---
## Latihan Mandiri

Jalankan ulang cell pembuatan `SparkSession` dan kedua DataFrame (`df_transaksi`, `df_produk`) dari awal modul sebelum mengerjakan latihan berikut.

In [37]:
# Persiapan ulang untuk latihan
spark = SparkSession.builder.appName("Latihan5").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

np.random.seed(7)
df_produk = spark.createDataFrame(pd.DataFrame(data_produk))
df_transaksi = spark.createDataFrame(pd.DataFrame(data_transaksi))
df_transaksi.createOrReplaceTempView("transaksi")
df_produk.createOrReplaceTempView("produk")
print("Siap untuk latihan.")

Siap untuk latihan.


**Soal 1.** Menggunakan DataFrame API (`join`), tampilkan seluruh transaksi kota `"Solo"` beserta nama `manager` dari kategorinya masing-masing.

In [38]:
df_gabung = df_transaksi.filter(df_transaksi.kota == "Solo").join(df_produk, on="kategori", how="left")
df_gabung.show(df_gabung.count(), truncate=False)

+----------------------+--------+----+----------+--------------+-------+
|kategori              |order_id|kota|pendapatan|target_bulanan|manager|
+----------------------+--------+----+----------+--------------+-------+
|Kesehatan & Kecantikan|O2      |Solo|452308    |25000000      |Dewi   |
|Kesehatan & Kecantikan|O15     |Solo|435461    |25000000      |Dewi   |
|Kesehatan & Kecantikan|O67     |Solo|309030    |25000000      |Dewi   |
|Kesehatan & Kecantikan|O68     |Solo|102276    |25000000      |Dewi   |
|Kesehatan & Kecantikan|O72     |Solo|478508    |25000000      |Dewi   |
|Kesehatan & Kecantikan|O88     |Solo|270598    |25000000      |Dewi   |
|Kesehatan & Kecantikan|O95     |Solo|273764    |25000000      |Dewi   |
|Elektronik            |O10     |Solo|72294     |50000000      |Andi   |
|Elektronik            |O12     |Solo|75566     |50000000      |Andi   |
|Elektronik            |O14     |Solo|459719    |50000000      |Andi   |
|Elektronik            |O21     |Solo|176776    |50

**Soal 2.** Menggunakan **window function**, tentukan transaksi dengan `pendapatan` **tertinggi** (peringkat 1 saja) di **setiap kota** (bukan kategori). tidak boleh menggunakan `rank()`.

In [39]:
window_spec = Window.partitionBy("kota").orderBy(col("pendapatan").desc())

df_ranked = df_transaksi.withColumn("peringkat", row_number().over(window_spec))

df_ranked.filter(col("peringkat") == 1).show(truncate=False)

+--------+------------+--------+----------+---------+
|order_id|kategori    |kota    |pendapatan|peringkat|
+--------+------------+--------+----------+---------+
|O96     |Fashion     |Magelang|498770    |1        |
|O147    |Rumah Tangga|Semarang|497370    |1        |
|O286    |Rumah Tangga|Solo    |497826    |1        |
+--------+------------+--------+----------+---------+



**Soal 3.** Menggunakan **Spark SQL** (bukan DataFrame API), tulis kueri untuk menghitung rata-rata `pendapatan` per `kota`, urutkan dari tertinggi.

In [40]:
hasil_sql = spark.sql('''
    SELECT kota, AVG(pendapatan) AS rata_rata
    FROM transaksi
    GROUP BY kota
    ORDER BY rata_rata DESC
''')
hasil_sql.show()

[Stage 18:>                                                         (0 + 3) / 3]

+--------+------------------+
|    kota|         rata_rata|
+--------+------------------+
|    Solo|278926.25925925927|
|Semarang|          267303.8|
|Magelang| 263705.6568627451|
+--------+------------------+



Soal 4 (Refleksi singkat).** Dalam 2-3 kalimat: menurut anda, dalam situasi seperti apa anda akan lebih memilih menulis Spark SQL dibanding DataFrame API pada pekerjaan anda nanti? Tulis jawaban pada markdown cell di bawah ini.

Saya lebih memilih menggunakan Spark SQL ketika pekerjaan yang dilakukan berupa pengolahan dan analisis data yang membutuhkan banyak operasi seperti SELECT, GROUP BY, JOIN, dan ORDER BY. Menurut saya, Spark SQL lebih mudah dibaca dan dipahami ketika proses pengolahan data cukup kompleks, terutama jika sudah terbiasa dengan bahasa SQL.

---
## TUGAS MANDIRI (Dikerjakan Selama 1 Minggu)

> **Tenggat waktu:** dikumpulkan paling lambat **sebelum Pertemuan 6 dimulai**.
> **Sifat tugas:** individu.

### Konteks / Skenario

Manajemen platform e-commerce meminta dibuatkan **dashboard performa cabang toko** yang menggabungkan data transaksi (yang sudah ada di HDFS sejak Pertemuan 3-4) dengan data referensi target penjualan tiap cabang. Anda ditugaskan menyiapkan analisis ini menggunakan kombinasi **join, window function, dan Spark SQL** — persis seperti yang dipelajari hari ini.

### Menyiapkan Dataset

Jalankan cell berikut untuk membuat **dua tabel** dan mengunggah tabel transaksi ke HDFS (tabel target cukup dibuat langsung sebagai Spark DataFrame, karena berukuran kecil dan jarang berubah — praktik umum untuk tabel referensi/*dimension table*).

In [41]:
import numpy as np
import pandas as pd

# Tabel 1: Target & PIC per cabang kota (tabel referensi, dibuat langsung sebagai DataFrame)
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}

# Tabel 2: Data transaksi (disimpan sebagai CSV, lalu diunggah ke HDFS)
np.random.seed(55)
n = 500
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]
data_transaksi_t5 = {
    "order_id": [f"TRX-{i}" for i in range(n)],
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 10, size=n),
    "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000], size=n),
}
pd.DataFrame(data_transaksi_t5).to_csv("transaksi_tugas5.csv", index=False)

!hdfs dfs -mkdir -p /user/mahasiswa/tugas5
!hdfs dfs -put -f transaksi_tugas5.csv /user/mahasiswa/tugas5/
print("Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/mahasiswa/tugas5/transaksi_tugas5.csv")
print("Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.")

Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/mahasiswa/tugas5/transaksi_tugas5.csv
Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.


### Instruksi Pengerjaan

Buat notebook baru **`Tugas5_[NPM]_[Nama Lengkap].ipynb`**, buat `SparkSession`, lalu:
1. Baca `transaksi_tugas5.csv` **dari HDFS** menjadi `df_transaksi`, tambahkan kolom `pendapatan` (`unit_terjual x harga_satuan`).
2. Buat `df_target` dari dictionary `data_target_cabang` di atas

Kerjakan bagian **A sampai D** berikut:

---

**A. Join & Perbandingan Target** *(bobot 25%)*

Ringkas total `pendapatan` per `kota` dari `df_transaksi`, lalu **join** dengan `df_target`. Tambahkan kolom `pencapaian_persen`. Urutkan hasil dari pencapaian tertinggi.

**B. Window Function — Kategori Terlaris per Kota** *(bobot 25%)*

Menggunakan window function, tentukan **kategori dengan pendapatan tertinggi di setiap kota** (top-1 saja, gunakan `row_number()`).

**C. Spark SQL** *(bobot 25%)*

Daftarkan `df_transaksi` dan `df_target` sebagai *temporary view*, lalu **tulis satu kueri SQL** (bukan DataFrame API) yang menampilkan: `kota`, `pic_cabang`, dan jumlah transaksi (`COUNT`) di kota tersebut, diurutkan dari jumlah transaksi terbanyak.

**D. Kesimpulan** *(bobot 25%)*

Tulis pada markdown cell (**minimal 100 kata**): berdasarkan hasil bagian A dan B, **cabang mana yang berkinerja paling baik** dan **cabang mana yang paling perlu perhatian manajemen**? Sertakan angka-angka pendukung dari hasil analisis kalian, bukan opini tanpa dasar data.

---

### Ketentuan Pengumpulan

- Kumpulkan `Tugas5_[NPM]_[Nama Lengkap].ipynb` melalui Asisten Praktikum, paling lambat **1 minggu dari hari ini, pukul 23.59 WIB**.
- Pastikan Hadoop aktif dan seluruh cell sudah dijalankan (**Run All**) sebelum dikumpulkan.
- Bagian A & B **wajib** menggunakan DataFrame API; bagian C **wajib** menggunakan Spark SQL murni (`spark.sql(...)`).

### Rubrik Penilaian

| Bagian | Kriteria | Bobot |
|---|---|---|
| A. Join & Pencapaian Target | Join benar, kolom `pencapaian_persen` terhitung tepat | 25% |
| B. Window Function | Kategori terlaris per kota teridentifikasi dengan benar menggunakan `row_number()` | 25% |
| C. Spark SQL | Kueri SQL berjalan benar & menghasilkan output yang sesuai instruksi | 25% |
| D. Kesimpulan | Analisis berbasis data, jelas menyebutkan cabang terbaik & yang perlu perhatian | 25% |


In [42]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, count, row_number
from pyspark.sql.window import Window

# Jalankan ini dulu sebelum mengerjakan latihan di bawah
spark = SparkSession.builder.appName("Tugas5").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

df_transaksi = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/tugas5/transaksi_tugas5.csv", header=True, inferSchema=True)

df_transaksi = df_transaksi.withColumn("pendapatan", col("unit_terjual") * col("harga_satuan"))

print("Siap. Jumlah baris:", df_transaksi.count())

Siap. Jumlah baris: 500


In [43]:
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"]
}

df_target = spark.createDataFrame(pd.DataFrame(data_target_cabang))
df_target.show()

+----------+--------------+----------+
|      kota|target_bulanan|pic_cabang|
+----------+--------------+----------+
|  Magelang|      45000000|      Rani|
|Yogyakarta|      60000000|      Joko|
|  Semarang|      55000000|      Sari|
|      Solo|      40000000|      Bayu|
| Purworejo|      30000000|     Fitri|
+----------+--------------+----------+



In [44]:
#A. Join & Perbandingan Target

pendapatan_kota = df_transaksi.groupBy("kota").agg(
    sum("pendapatan").alias("total_pendapatan")
)

hasil_a = pendapatan_kota.join(
    df_target,
    on="kota",
    how="inner"
)

hasil_a = hasil_a.withColumn(
    "pencapaian_persen",
    col("total_pendapatan") / col("target_bulanan") * 100
)

hasil_a = hasil_a.orderBy(
    col("pencapaian_persen").desc()
)

hasil_a.select(
    "kota",
    "pic_cabang",
    "total_pendapatan",
    "target_bulanan",
    "pencapaian_persen"
).show()

+----------+----------+----------------+--------------+------------------+
|      kota|pic_cabang|total_pendapatan|target_bulanan| pencapaian_persen|
+----------+----------+----------------+--------------+------------------+
| Purworejo|     Fitri|        45650000|      30000000|152.16666666666669|
|      Solo|      Bayu|        33475000|      40000000|           83.6875|
|Yogyakarta|      Joko|        47275000|      60000000| 78.79166666666667|
|  Magelang|      Rani|        31650000|      45000000| 70.33333333333334|
|  Semarang|      Sari|        38175000|      55000000|  69.4090909090909|
+----------+----------+----------------+--------------+------------------+



In [45]:
#B. Window Function — Kategori Terlaris per Kota

pendapatan_kategori = df_transaksi.groupBy(
    "kota", "kategori"
).agg(
    sum("pendapatan").alias("total_pendapatan")
)

window_kota = Window.partitionBy("kota").orderBy(
    col("total_pendapatan").desc()
)

hasil_b = pendapatan_kategori.withColumn(
    "peringkat",
    row_number().over(window_kota)
)

hasil_b = hasil_b.filter(
    col("peringkat") == 1
)

hasil_b.select(
    "kota",
    "kategori",
    "total_pendapatan"
).show()

+----------+--------------------+----------------+
|      kota|            kategori|total_pendapatan|
+----------+--------------------+----------------+
|  Magelang|Kesehatan & Kecan...|         7275000|
| Purworejo|Kesehatan & Kecan...|        10075000|
|  Semarang|        Rumah Tangga|        11125000|
|      Solo|Kesehatan & Kecan...|         8425000|
|Yogyakarta|             Fashion|        13325000|
+----------+--------------------+----------------+



In [46]:
#C. Spark SQL

df_transaksi.createOrReplaceTempView("transaksi")
df_target.createOrReplaceTempView("target")

hasil_c = spark.sql("""
    SELECT
        transaksi.kota,
        target.pic_cabang,
        COUNT(transaksi.order_id) AS jumlah_transaksi
    FROM transaksi
    JOIN target
        ON transaksi.kota = target.kota
    GROUP BY transaksi.kota, target.pic_cabang
    ORDER BY jumlah_transaksi DESC
""")

hasil_c.show()

[Stage 39:>                                                         (0 + 3) / 3]

+----------+----------+----------------+
|      kota|pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
| Purworejo|     Fitri|             116|
|Yogyakarta|      Joko|             110|
|      Solo|      Bayu|              95|
|  Semarang|      Sari|              93|
|  Magelang|      Rani|              86|
+----------+----------+----------------+



## D. Kesimpulan

Berdasarkan hasil analisis pada bagian A, cabang yang memiliki kinerja paling baik adalah **[nama kota]** dengan total pendapatan sebesar **Rp[total pendapatan]** dari target bulanan sebesar **Rp[target]**. Pencapaian target cabang tersebut adalah sebesar **[persentase]%**. Sementara itu, cabang yang paling perlu mendapatkan perhatian manajemen adalah **[nama kota]**, dengan total pendapatan sebesar **Rp[total pendapatan]** dari target sebesar **Rp[target]**, sehingga pencapaiannya hanya sebesar **[persentase]%**.

Berdasarkan hasil bagian B, kategori dengan pendapatan tertinggi di cabang **[nama kota]** adalah **[kategori]**, dengan total pendapatan sebesar **Rp[pendapatan]**. Pada cabang yang paling perlu mendapatkan perhatian, kategori dengan pendapatan tertingginya adalah **[kategori]**, dengan total pendapatan sebesar **Rp[pendapatan]**. Hasil tersebut menunjukkan bahwa pencapaian target setiap cabang berbeda-beda. Data kategori terlaris juga dapat memberikan gambaran mengenai kategori yang memberikan kontribusi pendapatan terbesar di setiap kota. Oleh karena itu, hasil analisis ini dapat digunakan sebagai dasar untuk mengevaluasi pencapaian penjualan dan menentukan cabang serta kategori yang perlu dipertahankan atau ditingkatkan.

In [47]:
hasil_a.show()
hasil_b.show()

+----------+----------------+--------------+----------+------------------+
|      kota|total_pendapatan|target_bulanan|pic_cabang| pencapaian_persen|
+----------+----------------+--------------+----------+------------------+
| Purworejo|        45650000|      30000000|     Fitri|152.16666666666669|
|      Solo|        33475000|      40000000|      Bayu|           83.6875|
|Yogyakarta|        47275000|      60000000|      Joko| 78.79166666666667|
|  Magelang|        31650000|      45000000|      Rani| 70.33333333333334|
|  Semarang|        38175000|      55000000|      Sari|  69.4090909090909|
+----------+----------------+--------------+----------+------------------+

+----------+--------------------+----------------+---------+
|      kota|            kategori|total_pendapatan|peringkat|
+----------+--------------------+----------------+---------+
|  Magelang|Kesehatan & Kecan...|         7275000|        1|
| Purworejo|Kesehatan & Kecan...|        10075000|        1|
|  Semarang|       